# Compose Overview & YAML Syntax

Running a single container with `docker run` is manageable. Running five containers — a web server, an API, a database, a cache, and a background worker — with the right networks, volumes, environment variables, and restart policies is not. Docker Compose solves this by letting you define your entire multi-container application in a single YAML file and manage it with one command.

Topics:
- What Compose is and how it relates to `docker run`
- The `compose.yaml` file structure
- Services, networks, and volumes as top-level keys
- Service configuration: image, build, ports, environment, volumes, depends_on, healthcheck
- Variable substitution with `.env`
- The `docker compose` command set
- Compose file versions and the modern format

## 1. What is Docker Compose?

Docker Compose is a tool for defining and running multi-container applications. You describe the desired state of your application in a YAML file — which containers to run, how they connect, what storage they use — and Compose handles creating, starting, stopping, and removing everything together.

**Compose is not an orchestrator.** It runs on a single Docker host. For multi-host orchestration, you use Docker Swarm or Kubernetes. Compose is for development, testing, and simple single-host production deployments.

### `docker run` vs Compose

The same three-container stack expressed both ways:

```bash
# docker run — three commands, easy to get out of sync
docker network create appnet
docker run -d --name db    --network appnet -e POSTGRES_PASSWORD=secret postgres:16
docker run -d --name redis --network appnet redis:7-alpine
docker run -d --name web   --network appnet -p 8000:8000 --depends-on db myapp:latest
```

```yaml
# compose.yaml — one file, one command
services:
  db:
    image: postgres:16
    environment:
      POSTGRES_PASSWORD: secret
  redis:
    image: redis:7-alpine
  web:
    image: myapp:latest
    ports:
      - "8000:8000"
    depends_on:
      - db
      - redis
```

```bash
docker compose up -d
```

## 2. The Compose File

### File name

Compose looks for these file names in the current directory, in order:
1. `compose.yaml` ← preferred (modern)
2. `compose.yml`
3. `docker-compose.yaml` ← legacy name (Compose v1)
4. `docker-compose.yml`

You can specify a different file with `-f`:
```bash
docker compose -f compose.prod.yaml up -d
```

### Top-level structure

```yaml
# Optional: name gives the project a prefix (default: directory name)
name: myapp

services:    # required — defines the containers
  web: ...
  db:  ...

networks:    # optional — custom networks
  appnet: {}

volumes:     # optional — named volumes
  pgdata: {}
```

### The project name

Compose prefixes every resource it creates with the **project name**: `myapp_web_1`, `myapp_db_1`, `myapp_appnet`. This means multiple Compose stacks can run on the same host without name collisions. The default project name is the directory name containing `compose.yaml`.

## 3. Service Configuration

Each key under `services` is a service name. The service definition maps directly to `docker run` flags:

```yaml
services:
  web:
    # ── Image ────────────────────────────────────────────────────────────────
    image: nginx:alpine          # pull this image
    # OR build from a Dockerfile:
    build:
      context: ./app             # build context directory
      dockerfile: Dockerfile     # default — can omit
      args:
        BUILD_ENV: production

    # ── Container name ────────────────────────────────────────────────────────
    container_name: my-web       # override the generated name (disables scaling)

    # ── Ports ─────────────────────────────────────────────────────────────────
    ports:
      - "8080:80"                # host:container
      - "127.0.0.1:8443:443"    # interface-bound

    # ── Environment variables ─────────────────────────────────────────────────
    environment:
      NODE_ENV: production
      DB_HOST: db                # use service name — Compose DNS resolves it
      DB_PASSWORD: ${DB_PASSWORD} # from .env file or shell
    env_file:
      - .env                     # load additional vars from file

    # ── Volumes ───────────────────────────────────────────────────────────────
    volumes:
      - appdata:/app/data        # named volume
      - ./config:/app/config:ro  # bind mount (relative path — resolved from compose file)

    # ── Networks ──────────────────────────────────────────────────────────────
    networks:
      - frontend
      - backend

    # ── Restart policy ────────────────────────────────────────────────────────
    restart: unless-stopped

    # ── Resource limits ───────────────────────────────────────────────────────
    deploy:
      resources:
        limits:
          memory: 512m
          cpus: '1.0'

    # ── Dependencies ──────────────────────────────────────────────────────────
    depends_on:
      db:
        condition: service_healthy  # wait for db health check to pass

    # ── Health check ──────────────────────────────────────────────────────────
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8080/health"]
      interval: 30s
      timeout: 5s
      retries: 3
      start_period: 10s

    # ── Command override ──────────────────────────────────────────────────────
    command: ["gunicorn", "--bind", "0.0.0.0:8000", "app:create_app()"]
    entrypoint: ["/docker-entrypoint.sh"]
```

## 4. Networks in Compose

### Default network

If you don't define any networks, Compose creates a single network named `<project>_default` and attaches all services to it. Every service can reach every other service by service name.

### Custom networks

```yaml
services:
  proxy:
    image: nginx:alpine
    networks: [frontend]
  api:
    image: myapi
    networks: [frontend, backend]
  db:
    image: postgres:16
    networks: [backend]

networks:
  frontend: {}              # user-defined bridge, Docker picks subnet
  backend:
    internal: true          # no outbound internet
    driver_opts:
      com.docker.network.bridge.name: br-backend  # custom bridge name
```

### Using an existing network

```yaml
networks:
  existing-net:
    external: true          # use a network created outside of Compose
    name: my-external-net
```

## 5. Volumes in Compose

```yaml
services:
  db:
    image: postgres:16
    volumes:
      - pgdata:/var/lib/postgresql/data
      - ./init.sql:/docker-entrypoint-initdb.d/init.sql:ro

volumes:
  pgdata: {}              # named volume — Docker manages it

  # Volume with a custom driver
  nfs-data:
    driver: local
    driver_opts:
      type: nfs
      o: addr=192.168.1.1,rw
      device: :/exports/data

  # Use an externally created volume
  external-vol:
    external: true
    name: my-existing-volume
```

**Relative bind mount paths** in the `volumes` list are resolved relative to the `compose.yaml` file location — not the current working directory. This makes Compose files portable.

## 6. `depends_on` and Health Checks

`depends_on` controls **startup order**, not application readiness by default.

```yaml
# Basic: web starts after db, but db might not be ready to accept connections yet
services:
  web:
    depends_on:
      - db
```

For true readiness, combine with `condition: service_healthy` and a `healthcheck` on the dependency:

```yaml
services:
  web:
    depends_on:
      db:
        condition: service_healthy   # waits for db's healthcheck to pass
      redis:
        condition: service_started   # just waits for redis to start (no healthcheck)

  db:
    image: postgres:16
    environment:
      POSTGRES_PASSWORD: secret
    healthcheck:
      test: ["CMD", "pg_isready", "-U", "postgres"]
      interval: 5s
      timeout: 3s
      retries: 5
      start_period: 10s
```

`condition` options:
- `service_started` — container has started (default if just listing service name)
- `service_healthy` — healthcheck has passed
- `service_completed_successfully` — for one-shot init containers (exit 0)

## 7. Variable Substitution and `.env`

Hardcoding passwords and environment-specific values in `compose.yaml` is bad practice. Compose supports variable substitution:

```yaml
services:
  db:
    image: postgres:${POSTGRES_VERSION:-16}   # default to 16 if not set
    environment:
      POSTGRES_PASSWORD: ${DB_PASSWORD}       # required — Compose warns if unset
      POSTGRES_DB: ${DB_NAME:-appdb}          # default to 'appdb'
```

### `.env` file

Compose automatically loads a `.env` file from the same directory as `compose.yaml`:

```bash
# .env
POSTGRES_VERSION=16
DB_PASSWORD=supersecret
DB_NAME=myapp
```

Priority order (highest to lowest):
1. Shell environment variables
2. `.env` file
3. Default in the Compose file (`:-default`)

**Never commit `.env` to version control** if it contains secrets. Add it to `.gitignore`. Commit a `.env.example` with placeholder values instead.

```bash
# See resolved variable values before running
docker compose config
```

## 8. The `docker compose` Command Set

```bash
# Start all services (build if needed, detached)
docker compose up -d

# Start and force rebuild images
docker compose up -d --build

# Stop all services (containers remain)
docker compose stop

# Stop and remove containers, networks (volumes preserved)
docker compose down

# Stop and remove everything including volumes
docker compose down -v

# Stop and remove everything including images
docker compose down --rmi all

# View running services
docker compose ps

# Stream logs from all services
docker compose logs -f

# Stream logs from one service
docker compose logs -f web

# Run a one-off command in a service container
docker compose run --rm web python manage.py migrate

# Execute a command in a running service container
docker compose exec web bash

# Scale a service (multiple instances)
docker compose up -d --scale api=3

# Restart one service
docker compose restart web

# Pull latest images without starting
docker compose pull

# Show resolved config (with variable substitution applied)
docker compose config

# Show service dependency graph
docker compose config --services
```

## 9. Complete Example: Web App + Postgres + Redis

In [ ]:
import os, textwrap

os.makedirs("/tmp/compose-demo", exist_ok=True)

# compose.yaml
with open("/tmp/compose-demo/compose.yaml", "w") as f:
    f.write(textwrap.dedent("""\
        name: webstack

        services:

          # ── Database ──────────────────────────────────────────────────────────
          db:
            image: postgres:16-alpine
            environment:
              POSTGRES_USER: app
              POSTGRES_PASSWORD: ${DB_PASSWORD:-changeme}
              POSTGRES_DB: appdb
            volumes:
              - pgdata:/var/lib/postgresql/data
            networks: [backend]
            restart: unless-stopped
            healthcheck:
              test: ["CMD", "pg_isready", "-U", "app"]
              interval: 5s
              timeout: 3s
              retries: 5
              start_period: 10s

          # ── Cache ─────────────────────────────────────────────────────────────
          redis:
            image: redis:7-alpine
            command: redis-server --appendonly yes
            volumes:
              - redisdata:/data
            networks: [backend]
            restart: unless-stopped
            healthcheck:
              test: ["CMD", "redis-cli", "ping"]
              interval: 5s
              timeout: 2s
              retries: 3

          # ── Application ───────────────────────────────────────────────────────
          web:
            image: nginx:alpine        # stand-in for the real app
            ports:
              - "${APP_PORT:-8080}:80"
            environment:
              DB_HOST: db
              REDIS_HOST: redis
            networks: [frontend, backend]
            restart: unless-stopped
            depends_on:
              db:
                condition: service_healthy
              redis:
                condition: service_healthy

        # ── Named volumes ──────────────────────────────────────────────────────
        volumes:
          pgdata: {}
          redisdata: {}

        # ── Networks ───────────────────────────────────────────────────────────
        networks:
          frontend: {}
          backend:
            internal: true
    """))

# .env
with open("/tmp/compose-demo/.env", "w") as f:
    f.write("DB_PASSWORD=devsecret\nAPP_PORT=8088\n")

print("compose.yaml and .env written")

In [ ]:
# Show resolved config — all variables substituted
!docker compose -f /tmp/compose-demo/compose.yaml \
    --project-directory /tmp/compose-demo \
    config --quiet && echo "Config is valid"

# Show what services are defined
!docker compose -f /tmp/compose-demo/compose.yaml \
    --project-directory /tmp/compose-demo \
    config --services

In [ ]:
# Start the stack
!docker compose -f /tmp/compose-demo/compose.yaml \
    --project-directory /tmp/compose-demo \
    up -d

import time; time.sleep(8)

!docker compose -f /tmp/compose-demo/compose.yaml \
    --project-directory /tmp/compose-demo \
    ps

In [ ]:
# Verify the web service is reachable on the mapped port from .env
!curl -s -o /dev/null -w "HTTP %{http_code} on port 8088\n" http://localhost:8088

In [ ]:
# Tear down — remove containers and networks; keep volumes
!docker compose -f /tmp/compose-demo/compose.yaml \
    --project-directory /tmp/compose-demo \
    down

# Tear down including volumes
!docker compose -f /tmp/compose-demo/compose.yaml \
    --project-directory /tmp/compose-demo \
    down -v

print("Stack removed")

## Summary

- **Docker Compose** defines a multi-container application in a single `compose.yaml` file. One `docker compose up -d` starts everything; `docker compose down` tears it all down.
- The file has three top-level sections: `services` (required), `networks` (optional), and `volumes` (optional).
- Each **service** maps to a `docker run` command. Key fields: `image` or `build`, `ports`, `environment`, `volumes`, `networks`, `restart`, `depends_on`, `healthcheck`.
- **Default network:** if no networks are defined, Compose creates one and connects all services to it — every service can reach every other by service name.
- **Custom networks:** declare them under `networks:` and assign services with `networks:` in the service definition. Use `internal: true` for backend isolation.
- **`depends_on` + `healthcheck`:** use `condition: service_healthy` to wait for a service to be genuinely ready, not just started.
- **Variable substitution:** `${VAR:-default}` in the Compose file reads from shell environment or `.env`. Use `docker compose config` to see resolved values.
- **`docker compose up/down/ps/logs/exec/run`** are the core commands. `down -v` removes volumes too. `up --build` forces a rebuild.